In [1]:
%%capture
!pip install -U langchain langchain-community langchain-huggingface langchain-qdrant langchain-text-splitters langchain-openai
!pip install -U qdrant-client sentence-transformers google-genai pydantic openai xmltodict
!pip install -U transformers accelerate bitsandbytes

In [2]:
import os
import glob
import pickle
# from google.colab import user_secrets

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, List
import re
import datetime

# from langchain_openai import ChatOpenAI
# from ragas.testset.generator import TestsetGenerator
# from ragas.testset.evolutions import simple, reasoning, multi_context
# from ragas.run_config import RunConfig

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
QDRANT_URL = user_secrets.get_secret('QDRANT_URL')
QDRANT_API_KEY = user_secrets.get_secret('QDRANT_API_KEY')
# GEMINI_API_KEY = user_secrets.get_secret('GEMINI_API_KEY')
OPENAI_API_KEY = user_secrets.get_secret('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret('OPENAI_API_KEY')
HF_TOKEN = user_secrets.get_secret('HF_TOKEN')
EMBEDDING_MODEL_ID = "intfloat/multilingual-e5-large"
EMBEDDING_DIM = 1024
# LLM_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
DATA_FOLDER_PATH = '/kaggle/input/datasets/jayawinata/medical-reports/processed-1'

current_date = datetime.datetime.now().strftime("%Y_%m_%d__%H%M")
COLLECTION_NAME = f"{current_date}_enriched_medical_reports"
PICKLE_BACKUP_PATH = f"{current_date}_medical_documents_enriched.pkl"

BATCH_SIZE = 1

In [6]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [7]:
def batch_generator(data: List, batch_size: int):
    """Yield successive n-sized batches from data."""
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]

In [8]:
# def get_section(md_text, start_pattern, max_length=5000):
#     """
#     Mengekstrak teks mulai dari pola judul yang ditemukan hingga batas karakter tertentu.
#     Sangat kebal terhadap dokumen OCR yang formatnya berantakan.
#     """
#     if not md_text:
#         return ""

#     start_match = re.search(start_pattern, md_text, re.IGNORECASE | re.MULTILINE)
#     if not start_match:
#         return ""

#     content_start = start_match.end()
#     section_content = md_text[content_start:]
#     section_content = section_content.strip()

#     if max_length is not None and len(section_content) > max_length:
#         section_content = section_content[:max_length]

#     section_content = section_content.replace("|", " ")
#     section_content = re.sub(r'-{2,}', ' ', section_content)
#     section_content = re.sub(r' {2,}', ' ', section_content)
#     section_content = "\n".join([line.rstrip() for line in section_content.splitlines()])
#     section_content = section_content.strip()

#     return section_content

# CHAR_LENGTH = 6000
# start_pattern = r"^(?:##\s*)?(?:BAB\s*[I1l\|]+\.?\s*)?(?:FORM\s*)?(?:NUTRITION(?:AL)?\s+CARE\s+PRO[CS]ESS?|NCP).*$"

In [9]:
# client = OpenAI(api_key=OPENAI_API_KEY)

In [10]:
# class Antropometri(BaseModel):
#     jenis_kelamin: str
#     usia: str
#     berat_badan: str
#     tinggi_badan: str
#     lingkar_kepala: Optional[str] = None
#     lingkar_lengan_atas: Optional[str] = None
#     indeks_massa_tubuh: Optional[str] = None

# class DataMedis(BaseModel):
#     anthropometric_data: Antropometri = Field(..., description="Objek data antropometri")
#     medical_diagnosis: str = Field(..., description="Diagnosis medis")

#     @property
#     def anthropometric_string(self) -> str:
#         # Mengubah object menjadi dictionary, hapus yang None, lalu gabung jadi string
#         data_dict = self.anthropometric_data.model_dump(exclude_none=True)
#         # Format: Key: Value, Key: Value
#         formatted_list = [f"{k.replace('_', ' ').title()}: {v}" for k, v in data_dict.items()]
#         return ", ".join(formatted_list)

In [11]:
# print(f"Sedang membaca file dari: {DATA_FOLDER_PATH}/*.md ...")

# documents = []
# files = glob.glob(os.path.join(DATA_FOLDER_PATH, "*.md"))

# if not files:
#     print("Tidak ada file .md ditemukan di folder tersebut!")
# else:
#     for file_path in files:
#         try:
#             filename = os.path.basename(file_path)
#             print(f"🔄 Processing: {filename}...", end=" ", flush=True)

#             with open(file_path, "r", encoding="utf-8") as f:
#                 text = f.read()

#             text_input = get_section(text, start_pattern, max_length=CHAR_LENGTH)

#             response = client.responses.parse(
#               model="gpt-4o-mini-2024-07-18",
#               input=[
#                   {"role": "system", "content": "Ekstrak informasi medis dari teks panjang yang diberikan"},
#                   {"role": "user", "content": text_input},
#               ],
#               text_format=DataMedis,
#             )

#             extracted_data = response.output_parsed

#             if extracted_data:
#                 meta_anthro = extracted_data.anthropometric_string
#                 meta_diag = extracted_data.medical_diagnosis
#                 meta_anthro_dict = extracted_data.anthropometric_data.model_dump(exclude_none=True)
#             else:
#                 meta_anthro = "Tidak Diketahui"
#                 meta_diag = "Tidak Diketahui"
#                 meta_anthro_dict = {}

#             print(f" Diagnosis {filename}: {meta_diag}")

#             # Create LangChain Document
#             doc = Document(
#                 page_content=text,
#                 metadata={
#                     "source": filename,
#                     "anthropometric_assessment": meta_anthro,
#                     "medical_diagnosis": meta_diag,
#                     "anthropometric_data": meta_anthro_dict
#                 }
#             )
#             documents.append(doc)
#             # print("anthro", meta_anthro)
#             # print('diag', meta_diag)
#             print("✅ OK")

#         except Exception as e:
#             print(f"\n   ❌ Gagal load {filename}: {e}")

In [12]:
# print(f"\nSedang memecah {len(documents)} dokumen...")

# headers_to_split_on = [
#     ("#", "Header 1"),
#     ("##", "Header 2"),
#     ("###", "Header 3"),
# ]

# text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

# chunked_documents = []

# for doc in documents:
#     # 1. Ambil teks mentahnya menggunakan .page_content lalu pecah
#     md_chunks = text_splitter.split_text(doc.page_content)

#     # 2. (Sangat Penting) Gabungkan metadata asli (seperti 'source' / nama file)
#     # dengan metadata header hasil pecahan
#     for chunk in md_chunks:
#         chunk.metadata.update(doc.metadata)

#     # 3. Masukkan ke keranjang akhir
#     chunked_documents.extend(md_chunks)

# enriched_documents = []

# print("Sedang memodifikasi konten dokumen dengan metadata...")

# for i, doc in enumerate(chunked_documents):
#     diagnosis = doc.metadata.get('medical_diagnosis', 'Tidak disebutkan')
#     anthro = doc.metadata.get('anthropometric_assessment', 'Data antropometri tidak tersedia')

#     new_content = f"""
#       INFO KLINIS PASIEN:
#       - Diagnosis Medis: {diagnosis}
#       - Assessment Fisik/Antropometri: {anthro}
#       \n\n
#       ISI LAPORAN:
#       {doc.page_content}
#     """
#     new_metadata = doc.metadata.copy()
#     new_metadata["chunk_id"] = i

#     # Buat objek Document baru dengan konten yang sudah di-enrich
#     # Metadata asli TETAP DISIMPAN agar nanti bisa kita panggil lagi di DataFrame
#     new_doc = Document(
#         page_content=new_content,
#         metadata=new_metadata
#     )

#     enriched_documents.append(new_doc)

# print(f"Selesai! {len(enriched_documents)} dokumen siap diproses Ragas.")

In [13]:
# print(f"\nMenyimpan backup chunks ke {PICKLE_BACKUP_PATH}...")

# with open(PICKLE_BACKUP_PATH, "wb") as f:
#     pickle.dump(enriched_documents, f)

In [14]:
PICKLE_BACKUP_PATH = "/kaggle/input/datasets/jayawinata/chunked-medical-reports/2026_04_29__1545_medical_documents_enriched.pkl"

if os.path.exists(PICKLE_BACKUP_PATH):
    with open(PICKLE_BACKUP_PATH, "rb") as f:
        enriched_documents = pickle.load(f)
    print(f"Berhasil memuat {len(enriched_documents)} dokumen dari {PICKLE_BACKUP_PATH}")
else:
    print(f"File {PICKLE_BACKUP_PATH} belum ada.")

Berhasil memuat 7237 dokumen dari /kaggle/input/datasets/jayawinata/chunked-medical-reports/2026_04_29__1545_medical_documents_enriched.pkl


In [15]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [16]:
print("\nMemulai proses Embedding & Upload ke Qdrant...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={
        "model_kwargs": {"device_map": "cuda", "quantization_config": bnb_config, "torch_dtype": torch.float16},
        "trust_remote_code": True,
        "token": HF_TOKEN,
    },
    encode_kwargs={
        "batch_size": BATCH_SIZE,
        "normalize_embeddings": True,
    }
)

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

if client.collection_exists(COLLECTION_NAME):
        print(f"   - Menghapus collection lama: {COLLECTION_NAME}")
        client.delete_collection(COLLECTION_NAME)
        
print(f"   - Membuat collection baru: {COLLECTION_NAME}")
client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIM,
            distance=models.Distance.COSINE
        ),
        metadata={
            "embedding_model": EMBEDDING_MODEL_ID,
            "created_at": datetime.datetime.now().isoformat()
        }
)



Memulai proses Embedding & Upload ke Qdrant...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

   - Membuat collection baru: 2026_06_11__1129_enriched_medical_reports


True

In [17]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [18]:
vector_store = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embedding_model,
    )

In [19]:
import time

In [20]:
print(f"Sedang mengupload {len(enriched_documents)} chunks ke Qdrant dalam batch {BATCH_SIZE}...")
    
total_batches = (len(enriched_documents) + BATCH_SIZE - 1) // BATCH_SIZE
for i, batch in enumerate(batch_generator(enriched_documents, BATCH_SIZE)):
    print(f"Mengupload batch {i+1}/{total_batches} ({len(batch)} documents)...")
    vector_store.add_documents(batch)
        
        # Periodic memory cleanup after each batch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nSELESAI! {len(enriched_documents)} chunks berhasil masuk ke Qdrant.")

🚀 Sedang mengupload 7237 chunks ke Qdrant dalam batch 1...
   📤 Mengupload batch 1/7237 (1 documents)...
   📤 Mengupload batch 2/7237 (1 documents)...
   📤 Mengupload batch 3/7237 (1 documents)...
   📤 Mengupload batch 4/7237 (1 documents)...
   📤 Mengupload batch 5/7237 (1 documents)...
   📤 Mengupload batch 6/7237 (1 documents)...
   📤 Mengupload batch 7/7237 (1 documents)...
   📤 Mengupload batch 8/7237 (1 documents)...
   📤 Mengupload batch 9/7237 (1 documents)...
   📤 Mengupload batch 10/7237 (1 documents)...
   📤 Mengupload batch 11/7237 (1 documents)...
   📤 Mengupload batch 12/7237 (1 documents)...
   📤 Mengupload batch 13/7237 (1 documents)...
   📤 Mengupload batch 14/7237 (1 documents)...
   📤 Mengupload batch 15/7237 (1 documents)...
   📤 Mengupload batch 16/7237 (1 documents)...
   📤 Mengupload batch 17/7237 (1 documents)...
   📤 Mengupload batch 18/7237 (1 documents)...
   📤 Mengupload batch 19/7237 (1 documents)...
   📤 Mengupload batch 20/7237 (1 documents)...
   📤 Mengu

KeyboardInterrupt: 

In [ ]:
!nvidia-smi